# Notebook 11: LLM System Design

## Why This Matters

LLMs have become the dominant paradigm for a wide range of applications—chatbots, code generation, search, summarization—and LLM system design has become a distinct and important interview topic at companies like Anthropic, OpenAI, Google DeepMind, Meta AI, and virtually every major tech company building AI products. Unlike classical ML systems, LLM systems have unique engineering challenges: KV cache management, speculative decoding, token streaming, retrieval-augmented generation (RAG), prompt management, and the cost-quality-latency tradeoff unique to autoregressive generation. Understanding these deeply—from the transformer architecture through to production serving optimization—distinguishes candidates with genuine LLM systems experience.

## 1. LLM Inference: How It Works Under the Hood

Understanding LLM inference at the systems level—not just the API level—is essential for system design. Autoregressive generation, KV caching, and the prefill vs decode distinction shape every architectural decision.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: Autoregressive generation phases
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('Autoregressive Generation: Prefill vs Decode', fontsize=11, fontweight='bold')

def box(ax, x, y, w, h, text, color, fontsize=9):
    r = FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.1",
                       facecolor=color, edgecolor='white', linewidth=1.5, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white', multialignment='center')

# Phase 1: Prefill
ax.add_patch(FancyBboxPatch((0.3, 6.8), 9.4, 2.8, boxstyle="round,pad=0.1",
                             facecolor='#2980B9', alpha=0.12, edgecolor='#2980B9', linewidth=2))
ax.text(5, 9.3, 'PHASE 1: PREFILL (compute-bound)', ha='center', fontsize=10,
        fontweight='bold', color='#2980B9')

tokens = ['"What', 'is', 'the', 'capital', 'of', 'France?"']
for i, tok in enumerate(tokens):
    box(ax, 0.4 + i * 1.5, 7.5, 1.3, 0.9, tok, '#2980B9', fontsize=8)

ax.annotate('All input tokens\nprocessed in parallel\n(one forward pass)', xy=(5, 7.0), fontsize=8.5,
            color='#2980B9', ha='center', style='italic')

box(ax, 2.5, 7.0, 5, 0.7, 'KV Cache Built from Prompt Tokens', '#1A5276', fontsize=8)

# Phase 2: Decode
ax.add_patch(FancyBboxPatch((0.3, 1.5), 9.4, 5.0, boxstyle="round,pad=0.1",
                             facecolor='#E74C3C', alpha=0.08, edgecolor='#E74C3C', linewidth=2))
ax.text(5, 6.2, 'PHASE 2: DECODE (memory-bandwidth-bound)', ha='center', fontsize=10,
        fontweight='bold', color='#E74C3C')

decode_steps = [
    ('Step 1', '"The"', 'KV cache: 6 tokens', '#E74C3C'),
    ('Step 2', '"capital"', 'KV cache: 7 tokens', '#C0392B'),
    ('Step 3', '"is"', 'KV cache: 8 tokens', '#A93226'),
    ('Step 4', '"Paris"', 'KV cache: 9 tokens', '#922B21'),
]

for i, (step, token, cache_info, color) in enumerate(decode_steps):
    y = 5.2 - i * 0.95
    box(ax, 0.4, y, 1.2, 0.7, step, color, fontsize=8)
    box(ax, 1.8, y, 1.5, 0.7, token, color, fontsize=8)
    ax.text(3.5, y + 0.35, f'← Generate 1 token; {cache_info}',
            fontsize=8, va='center', color='#555')
    ax.text(3.5, y + 0.05, f'   One forward pass per token (sequential)',
            fontsize=7.5, va='center', color='#888', style='italic')

ax.text(5, 1.8, 'Decode is N sequential forward passes (N = output length)',
        ha='center', fontsize=8.5, color='#E74C3C', fontweight='bold')

# Right: KV cache anatomy
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 10); ax2.axis('off')
ax2.set_title('KV Cache: The Key to Efficient Autoregressive Generation', fontsize=10, fontweight='bold')

# KV cache visualization
n_layers = 8
n_tokens = 12
cell_h = 0.55
cell_w = 0.7

ax2.text(5, 9.5, 'KV Cache (32 layers × n_tokens × 2 × head_dim × float16)',
         ha='center', fontsize=9, color='#333')

for layer in range(n_layers):
    y = 8.5 - layer * (cell_h + 0.05)
    ax2.text(0.3, y + cell_h/2, f'L{layer+1}', fontsize=7.5, va='center', ha='center', color='#666')
    for tok in range(n_tokens):
        x = 0.8 + tok * (cell_w + 0.02)
        color = '#2980B9' if tok < 6 else '#E74C3C' if tok < 10 else '#27AE60'
        rect = FancyBboxPatch((x, y), cell_w, cell_h,
                               boxstyle="round,pad=0.02", facecolor=color,
                               edgecolor='white', linewidth=1, alpha=0.7)
        ax2.add_patch(rect)

ax2.text(4.5, 4.0, 'Prompt\n(6 tokens)', ha='center', fontsize=8.5, color='#2980B9', fontweight='bold')
ax2.text(7.5, 4.0, 'Generated\n(4 tokens)', ha='center', fontsize=8.5, color='#E74C3C', fontweight='bold')
ax2.text(9.5, 4.0, 'Next\nto gen', ha='center', fontsize=8.5, color='#27AE60', fontweight='bold')

# Memory calculation
ax2.text(5, 3.3, 'KV Cache Memory = 2 × n_layers × n_heads × head_dim × seq_len × bytes_per_element',
         ha='center', fontsize=8, color='#333')

# Example: LLaMA-70B
n_layers_llama = 80
n_kv_heads = 8  # GQA
head_dim = 128
seq_len = 4096
bytes_per = 2  # float16
kv_cache_bytes = 2 * n_layers_llama * n_kv_heads * head_dim * seq_len * bytes_per
kv_cache_gb = kv_cache_bytes / 1e9

ax2.text(5, 2.7, f'LLaMA-70B (GQA): {kv_cache_gb:.1f} GB per sequence at 4096 tokens',
         ha='center', fontsize=9, color='#E74C3C', fontweight='bold')
ax2.text(5, 2.2, 'A100 80GB GPU: fits ~{:.0f} concurrent sequences'.format(80 / kv_cache_gb),
         ha='center', fontsize=9, color='#E67E22')
ax2.text(5, 1.6, 'Optimizations: PagedAttention (vLLM), GQA, MQA reduce KV cache size',
         ha='center', fontsize=9, color='#27AE60', style='italic')

plt.tight_layout()
plt.savefig('llm_inference.png', dpi=120, bbox_inches='tight')
plt.show()

print("Key LLM inference facts:")
print("  Prefill: processes all input tokens in ONE parallel forward pass")
print("  Decode:  generates ONE token per forward pass (N passes for N tokens)")
print("  KV Cache: avoids recomputing attention keys/values for previous tokens")
print("  Bottleneck: decode is memory-bandwidth-bound, not compute-bound")
print(f"  LLaMA-70B KV cache: {kv_cache_gb:.1f} GB per 4096-token sequence (with GQA)")

## 2. LLM Serving Optimizations

Efficient LLM serving requires specialized techniques that don't apply to classical ML. These optimizations are the difference between serving 1 request/second and 100 requests/second on the same hardware.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

optimizations = {
    'Continuous Batching (vLLM)': {
        'problem': 'Static batching: must wait for all sequences in batch to finish before starting new ones',
        'solution': 'Dynamically insert new requests as sequences finish — no idle GPU time',
        'speedup': '2-4×',
        'tradeoff': 'Sequence length variance within batch, complexity',
        'who_uses': 'vLLM, TGI (HuggingFace), SGLang',
    },
    'PagedAttention (vLLM)': {
        'problem': 'KV cache allocated as contiguous blocks — causes fragmentation, limits parallelism',
        'solution': 'Virtual memory-style paging for KV cache: non-contiguous pages shared across sequences',
        'speedup': '2-4× throughput via higher batch sizes',
        'tradeoff': 'Complexity, page management overhead',
        'who_uses': 'vLLM',
    },
    'Speculative Decoding': {
        'problem': 'Decode is sequential (1 token/forward pass) — low GPU utilization',
        'solution': 'Draft model generates K tokens fast; target model verifies in one pass\n(verify K draft tokens in parallel instead of one at a time)',
        'speedup': '2-3× on CPU/medium GPU',
        'tradeoff': 'Requires compatible draft model; overhead when acceptance low',
        'who_uses': 'Google, Anthropic, Deepmind (Medusa)',
    },
    'Flash Attention': {
        'problem': 'Standard attention reads/writes full attention matrix to HBM — memory-bandwidth bound',
        'solution': 'Tiled computation that keeps data in SRAM; 2-4× faster, 5-20× memory reduction',
        'speedup': '2-4× for prefill phase',
        'tradeoff': 'Only on CUDA, complex kernel',
        'who_uses': 'Essentially all production LLM deployments',
    },
    'Quantization (INT8/INT4)': {
        'problem': 'Large model weights require many GPUs → high cost',
        'solution': 'Reduce weight precision: FP16 → INT8 or INT4 (2× or 4× smaller)',
        'speedup': '1.5-2× on memory-bandwidth-bound decode',
        'tradeoff': 'Small perplexity degradation; accuracy-sensitive tasks affected',
        'who_uses': 'Most production deployments (GPTQ, AWQ, GGUF)',
    },
    'Tensor Parallelism': {
        'problem': 'Model too large for single GPU',
        'solution': 'Split weight matrices across GPUs (column-wise / row-wise); Megatron-LM approach',
        'speedup': 'Enables models that would OOM on single GPU',
        'tradeoff': 'All-reduce synchronization overhead; best with high-bandwidth interconnect (NVLink)',
        'who_uses': 'All large model deployments (vLLM, TGI, Ray)',
    },
}

for opt, details in optimizations.items():
    print(f"\n{'='*65}")
    print(f"  {opt}")
    print(f"{'='*65}")
    print(f"  Problem:   {details['problem'][:80]}")
    print(f"  Solution:  {details['solution'][:80]}")
    print(f"  Speedup:   {details['speedup']}")
    print(f"  Tradeoff:  {details['tradeoff']}")
    print(f"  Used by:   {details['who_uses']}")

## 3. Retrieval-Augmented Generation (RAG) System Design

RAG combines LLM generation with retrieval from a knowledge base, addressing the knowledge cutoff and hallucination problems. It's one of the most common LLM system design interview questions.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16); ax.set_ylim(0, 10); ax.axis('off')
ax.set_title('RAG System Architecture: Offline + Online Paths', fontsize=14, fontweight='bold')

def box(ax, x, y, w, h, text, color, fontsize=8.5):
    r = FancyBboxPatch((x,y), w, h, boxstyle="round,pad=0.1",
                       facecolor=color, edgecolor='white', linewidth=1.8, alpha=0.9)
    ax.add_patch(r)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='white', multialignment='center')

def arr(ax, x1, y1, x2, y2, label='', color='#666'):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.8))
    if label:
        ax.text((x1+x2)/2+0.1, (y1+y2)/2+0.15, label, fontsize=7.5, ha='center', color=color)

# OFFLINE: Document Ingestion
ax.text(8, 9.6, '─── OFFLINE: DOCUMENT INGESTION ───', ha='center', fontsize=10,
        color='#8E44AD', fontweight='bold')

box(ax, 0.3, 8.0, 3, 1.2, 'Source Documents\n(PDFs, web, DB, code)', '#7F8C8D')
box(ax, 4.0, 8.0, 3, 1.2, 'Chunking\n(512-1024 tokens)', '#E67E22')
box(ax, 7.7, 8.0, 3, 1.2, 'Embedding Model\n(text-embedding-3-large)', '#8E44AD')
box(ax, 11.4, 8.0, 4.2, 1.2, 'Vector Store\n(Pinecone/Weaviate/pgvector)', '#6C3483')

arr(ax, 3.3, 8.6, 4.0, 8.6)
arr(ax, 7.0, 8.6, 7.7, 8.6)
arr(ax, 10.7, 8.6, 11.4, 8.6)

# ONLINE: Query + Generate
ax.text(8, 7.2, '─── ONLINE: RETRIEVAL + GENERATION ─── (<2 seconds target) ───', ha='center',
        fontsize=10, color='#2980B9', fontweight='bold')

box(ax, 0.3, 5.5, 2.8, 1.2, 'User Query', '#7F8C8D')
box(ax, 3.5, 5.5, 3, 1.2, 'Query Embedding\n(same model as ingestion)', '#2980B9')
box(ax, 7.0, 5.5, 3, 1.2, 'Vector Search\n(top-K chunks, K=5-20)', '#3498DB')
box(ax, 10.5, 5.5, 5, 1.2, 'Context Window\n(system prompt + chunks + query)', '#1A5276')

arr(ax, 3.1, 6.1, 3.5, 6.1)
arr(ax, 6.5, 6.1, 7.0, 6.1)
arr(ax, 10.0, 6.1, 10.5, 6.1)

# Connect vector store to search
arr(ax, 13.5, 8.0, 8.5, 6.7, 'lookup', '#6C3483')

box(ax, 4.5, 3.5, 7, 1.5, 'LLM Generation\n(GPT-4 / Claude / LLaMA)\nGrounded in retrieved context', '#E74C3C')
arr(ax, 13, 5.5, 9.5, 5.0)
arr(ax, 8, 5.5, 8, 5.0)

box(ax, 4.5, 1.5, 7, 1.2, 'Response\n(grounded + cited)', '#27AE60')
arr(ax, 8, 3.5, 8, 2.7)

# Key design choices annotations
design_choices = [
    (0.3, 4.5, 'Chunk size:', '512-1024 tokens. Smaller = more precise retrieval, larger = more context.'),
    (0.3, 3.8, 'K (top-K):', '5-20 chunks. More = richer context but may dilute relevance.'),
    (0.3, 3.1, 'Reranker:', 'Optional: cross-encoder reranks top-K for higher precision.'),
    (0.3, 2.4, 'Hybrid search:', 'BM25 (keyword) + dense (semantic) for better recall.'),
]

for x, y, label, detail in design_choices:
    ax.text(x, y, label, fontsize=8.5, fontweight='bold', color='#333')
    ax.text(x + 1.5, y, detail, fontsize=8, color='#666')

plt.tight_layout()
plt.savefig('rag_architecture.png', dpi=120, bbox_inches='tight')
plt.show()

print("RAG System Design Key Decisions:")
decisions = [
    ("Embedding model", "Match ingestion and query embedding models (must be identical)"),
    ("Chunk size", "512-1024 tokens; overlap by ~10% to avoid cutting context boundaries"),
    ("Vector store", "pgvector (SQL+vectors), Pinecone (managed), Weaviate (multimodal)"),
    ("K retrieval", "5-20; more = higher recall but costs tokens and may dilute signal"),
    ("Reranker", "Cross-encoder (e.g., Cohere Rerank) re-scores top-50 to get precise top-5"),
    ("Hybrid search", "BM25 + dense vector: lexical match + semantic match → best recall"),
    ("Context length", "LLM's context window limits how many chunks you can include"),
]
for decision, rationale in decisions:
    print(f"  {decision}: {rationale}")

## 4. LLM Cost-Quality-Latency Tradeoffs

LLM serving has a three-way tradeoff: cost, quality, and latency. Every production LLM system must navigate this triangle based on the use case.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model tier comparison (approximate 2024/2025 figures)
models = [
    {'name': 'GPT-4o', 'params_B': 200, 'quality': 95, 'latency_ms': 800,
     'cost_per_1M_tokens': 5.0, 'color': '#E74C3C', 'use_case': 'Complex reasoning'},
    {'name': 'Claude 3.5 Sonnet', 'params_B': 100, 'quality': 93, 'latency_ms': 600,
     'cost_per_1M_tokens': 3.0, 'color': '#E67E22', 'use_case': 'Code, analysis'},
    {'name': 'LLaMA 3 70B', 'params_B': 70, 'quality': 85, 'latency_ms': 400,
     'cost_per_1M_tokens': 0.9, 'color': '#F39C12', 'use_case': 'General purpose, self-hosted'},
    {'name': 'GPT-4o mini', 'params_B': 8, 'quality': 78, 'latency_ms': 200,
     'cost_per_1M_tokens': 0.15, 'color': '#27AE60', 'use_case': 'Classification, simple tasks'},
    {'name': 'LLaMA 3 8B', 'params_B': 8, 'quality': 70, 'latency_ms': 100,
     'cost_per_1M_tokens': 0.05, 'color': '#2980B9', 'use_case': 'Low-latency, cost-sensitive'},
    {'name': 'Phi-3 mini', 'params_B': 3.8, 'quality': 65, 'latency_ms': 50,
     'cost_per_1M_tokens': 0.02, 'color': '#8E44AD', 'use_case': 'Edge, mobile'},
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
for m in models:
    ax.scatter(m['cost_per_1M_tokens'], m['quality'], s=150, c=m['color'],
               zorder=5, edgecolors='white', linewidth=2)
    ax.annotate(m['name'], (m['cost_per_1M_tokens'], m['quality']),
                xytext=(8, 5), textcoords='offset points', fontsize=8)

ax.set_xscale('log')
ax.set_xlabel('Cost per 1M output tokens (log scale, USD)', fontsize=10)
ax.set_ylabel('Quality Score (relative)', fontsize=10)
ax.set_title('Model Quality vs Cost\n(Efficiency frontier)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)

# Efficiency frontier
costs = sorted([m['cost_per_1M_tokens'] for m in models])
qualities = [m['quality'] for m in sorted(models, key=lambda x: x['cost_per_1M_tokens'])]
ax.plot(costs, qualities, 'k--', alpha=0.3, linewidth=1.5)

ax2 = axes[1]
for m in models:
    ax2.scatter(m['latency_ms'], m['quality'], s=150, c=m['color'],
                zorder=5, edgecolors='white', linewidth=2, label=m['name'])
    ax2.annotate(m['name'], (m['latency_ms'], m['quality']),
                 xytext=(8, 5), textcoords='offset points', fontsize=8)

ax2.axvline(x=200, color='orange', linestyle='--', alpha=0.7, label='200ms SLA')
ax2.axvline(x=1000, color='red', linestyle='--', alpha=0.7, label='1s SLA')
ax2.set_xlabel('TTFT + Generation Latency (ms)', fontsize=10)
ax2.set_ylabel('Quality Score (relative)', fontsize=10)
ax2.set_title('Model Quality vs Latency', fontsize=11, fontweight='bold')
ax2.legend(fontsize=7, loc='lower right')
ax2.grid(True, alpha=0.3)

plt.suptitle('LLM Model Selection: Cost-Quality-Latency Triangle', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('llm_tradeoffs.png', dpi=120, bbox_inches='tight')
plt.show()

print("Model selection heuristics:")
heuristics = [
    ("Chatbot, customer-facing", "GPT-4o / Claude 3.5 Sonnet — quality matters, user tolerance for latency"),
    ("Internal tool, moderate complexity", "LLaMA 3 70B (self-hosted) or GPT-4o mini — cost-quality balance"),
    ("Classification / routing", "GPT-4o mini / LLaMA 3 8B — cheap, fast enough"),
    ("High QPS, cost-sensitive", "Self-hosted LLaMA 3 8B + quantization (INT4/INT8)"),
    ("Edge / mobile", "Phi-3 mini or LLaMA 3 8B quantized to 4-bit"),
    ("Complex multi-step reasoning", "GPT-4o or Claude 3.5 Sonnet + chain-of-thought"),
]
for use_case, recommendation in heuristics:
    print(f"  {use_case}: {recommendation}")

## 5. LLM System Design Interview: Key Topics

A comprehensive LLM system design interview covers the full stack from prompt engineering through to production monitoring. Here's a structured summary of the most commonly asked topics.

In [ ]:
llm_system_topics = {
    "Inference Architecture": [
        "Prefill vs decode phase distinction",
        "KV cache: what it is, why it matters, memory cost",
        "Continuous batching (vLLM) vs static batching",
        "PagedAttention: virtual memory for KV cache",
        "Speculative decoding: draft model + verification",
        "Tensor parallelism for models > single GPU",
    ],
    "RAG Design": [
        "Chunking strategy (size, overlap, semantic chunking)",
        "Embedding models (match ingestion and query)",
        "Vector store selection (pgvector, Pinecone, Weaviate, Qdrant)",
        "Hybrid search (BM25 + dense)",
        "Reranking (cross-encoder for precision)",
        "Context window management (which chunks to include)",
    ],
    "Cost Optimization": [
        "Model tiering (route simple queries to smaller models)",
        "Prompt caching (cache common system prompts)",
        "Quantization (INT8/INT4): 2-4× cheaper",
        "Batching requests (higher GPU utilization)",
        "Output length control (max_tokens, early stopping)",
    ],
    "Latency Optimization": [
        "Streaming (token-by-token streaming improves perceived latency)",
        "TTFT vs TBT: time-to-first-token vs tokens-between-tokens",
        "Flash Attention (faster prefill)",
        "GPU memory hierarchy (HBM vs SRAM)",
        "Prompt caching reduces prefill time for repeated prefixes",
    ],
    "Reliability & Safety": [
        "Output validation (schema checking, guardrails)",
        "Hallucination detection (RAG grounding, citation)",
        "Rate limiting and abuse prevention",
        "Prompt injection defenses",
        "PII detection and redaction in inputs",
    ],
    "Evaluation": [
        "LLM-as-judge (GPT-4 evaluating responses)",
        "Human evaluation pipelines",
        "RAG evaluation: context relevance, faithfulness, answer relevance",
        "Regression testing: prompt changes vs baseline",
    ],
}

for category, items in llm_system_topics.items():
    print(f"\n{'='*60}")
    print(f"  {category}")
    print(f"{'='*60}")
    for item in items:
        print(f"  • {item}")

## Summary

| Concept | Key Point | Interview Signal |
|---------|-----------|------------------|
| Prefill vs decode | Prefill: parallel, compute-bound; Decode: sequential, memory-bandwidth-bound | Can explain why decode is the bottleneck |
| KV cache | Avoids recomputing attention; memory = 2 × layers × heads × dim × seq_len × bytes | Can estimate memory for a given model |
| Continuous batching | New requests inserted as sequences finish — higher GPU utilization | Knows the baseline (static batching) to compare against |
| PagedAttention | Virtual memory for KV cache; enables higher concurrency | Used in vLLM, know why it matters |
| Speculative decoding | Draft model + target model verification; 2-3× speedup | Know the acceptance rate tradeoff |
| RAG design | Chunk → embed → store → retrieve → generate | Can walk through all design choices |
| Model tiering | Route cheap queries to small models; complex to large | Cost optimization mindset |
| Quantization | INT8: 2× smaller, minimal quality loss; INT4: 4× smaller, some degradation | Know the accuracy-cost tradeoff |